# 00 · Pipeline overview & configuration

This is the entry point for the **MILK** (Multi-Instance Learning Kit) notebook family.
Each notebook launches one step of the pipeline and can be run on its own:

| Notebook | Step | What it does |
|----------|------|--------------|
| `00_overview_and_config.ipynb` | Setup | Loads & inspects the YAML config (this notebook) |
| `01_data_preprocessing.ipynb` | Data preprocessing | Builds bags, clusters conformers, scales features via `MILDataModule.setup()` |
| `02_model_construction.ipynb` | Model construction | Builds the MIL model (embedder → aggregator → predictor) |
| `03_model_training.ipynb` | Training & evaluation | Runs Stage 1 (train/val/test on the predefined split) and optional Stage 2 (final fit + test) |

The data uses a **predefined split** (`split` column: 0=train, 1=val, 2=test) —
no cross-validation. All four notebooks read the **same** `CONFIG_PATH`, so
changing the config in one place keeps every step consistent. Under the hood
these notebooks call the exact same classes as `poetry run milk -c <config>` —
nothing is re-implemented.

In [ ]:
# --- Bootstrap: make the notebook run from anywhere ---
import os, sys, logging
from pathlib import Path

# Locate the project root (folder that contains the `ppl` package).
here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [here, *here.parents] if (p / 'ppl' / '__init__.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    # Fallback: this notebook lives in <root>/notebooks/
    PROJECT_ROOT = Path('__file__' in globals() and __file__ or '.').resolve().parent.parent

os.chdir(PROJECT_ROOT)                       # pipeline writes outputs relative to cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
print('Project root:', PROJECT_ROOT)

In [ ]:
# Path to the experiment YAML. Edit this to point at a different config.
CONFIG_PATH = PROJECT_ROOT / 'ppl/utils/experiment_configs/nos_regression_experiment_config.yaml'
assert CONFIG_PATH.exists(), f'Config not found: {CONFIG_PATH}'
print('Using config:', CONFIG_PATH.relative_to(PROJECT_ROOT))

## Load the config

`PipelineConfig.from_yaml` validates the YAML and builds the three config
sections (`data`, `model`, `trainer`). `PipelineConfigManager` then applies
defaults/overrides exactly as the real pipeline does.

In [ ]:
from ppl.utils.modelling_configs.pipeline_config import PipelineConfig
from ppl.utils.pipeline.config_manager import PipelineConfigManager

cfg = PipelineConfig.from_yaml(CONFIG_PATH)
cfg_mgr = PipelineConfigManager(cfg)

print('task      :', cfg_mgr.task)
print('seed      :', cfg_mgr.seed)
print('log dir   :', cfg_mgr.log_save_dir)
print('experiment:', cfg_mgr.trainer_cfg.experiment_name)

### Data configuration

In [ ]:
import dataclasses as dc, pprint
pprint.pp(dc.asdict(cfg_mgr.data_cfg))

### Model configuration

In [ ]:
print('template        :', cfg_mgr.model_cfg.template)
print('embedder_type   :', cfg_mgr.model_cfg.embedder_type)
print('aggregator_type :', cfg_mgr.model_cfg.aggregator_type)
print('predictor_type  :', cfg_mgr.model_cfg.predictor_type)

### Trainer configuration

In [ ]:
print('max_epochs      :', cfg_mgr.trainer_cfg.max_epochs)
print('device          :', cfg_mgr.trainer_cfg.device)
print('precision       :', cfg_mgr.trainer_cfg.precision)
print('checkpoint_monitor:', cfg_mgr.trainer_cfg.checkpoint_monitor)
print('stage_2_launch  :', cfg_mgr.trainer_cfg.stage_2_launch)

---
Config looks good? Continue with **`01_data_preprocessing.ipynb`**.